# 01 — Problem Framing and Data Understanding

**Machine Learning-Based Network Intrusion Detection and Anomaly Classification**

This notebook establishes what we are predicting, from what, and what is wrong
with the data before any modelling decision is made.

| | |
|---|---|
| **Task** | Supervised binary classification |
| **Target** | `label` — 0 = benign, 1 = malicious |
| **Unit** | One bidirectional network flow record |
| **Dataset** | UNSW-NB15 (Moustafa & Slay, 2015) |
| **Constraint** | Flow statistics only — no payloads, IPs, ports or timestamps |

Full framing, pre-registered success criteria and risk analysis:
[`reports/problem_statement.md`](../reports/problem_statement.md).

In [1]:
import sys, warnings
from pathlib import Path

# Make the repository root importable no matter where Jupyter was launched from.
ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=FutureWarning)

from src import config
print(f"Repository root : {ROOT}")
print(f"Random seed     : {config.RANDOM_STATE}")

Repository root : E:\AIM\AIM AI ML Capstone\AI_Capstone_Network_Intrusion_Detection
Random seed     : 42


## 1. Acquire and verify the dataset

The loader does not trust the mirror it downloads from. It re-derives the
dataset's identity from the file contents: expected row counts, column count,
the binary encoding of `label`, and the rule that `attack_cat == "Normal"` if
and only if `label == 0`.

In [2]:
from src import data_loader

data_loader.download_dataset(verbose=True)
findings = data_loader.verify_dataset(strict=True, verbose=True)

display(Markdown("**Integrity findings**"))
for key, value in findings.items():
    if key != "problems":
        print(f"  {key:<32} {value}")

[data_loader] present    UNSW_NB15_training-set.csv  (32.3 MB)
[data_loader] present    UNSW_NB15_testing-set.csv  (15.4 MB)


[data_loader] integrity check: PASS


**Integrity findings**

  train_shape                      (175341, 45)
  test_shape                       (82332, 45)
  label_values                     [0, 1]
  label_attack_cat_agreement       1.0
  ok                               True


In [3]:
for filename, digest in data_loader.file_checksums().items():
    print(f"{filename:<32} sha256 {digest}")

UNSW_NB15_training-set.csv       sha256 bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675fd0339e07dfe0648fa
UNSW_NB15_testing-set.csv        sha256 734fe6642edf758f7c94d7d9149426b49d202fe8e7bf0bef47392489c3c0a559


### Interpretation

Both partitions match the row counts published by the dataset authors
(175,341 / 82,332) and carry the expected 45 columns. The
`label` / `attack_cat` agreement is exactly 1.0000 — which is precisely why
`attack_cat` must never be used as a predictor. It *is* the target, relabelled.

## 2. Structure

In [4]:
corpus = data_loader.load_corpus(verbose=True)

print(f"\nShape: {corpus.shape}")
print(f"Memory: {corpus.memory_usage(deep=True).sum() / 1e6:,.1f} MB")
display(corpus.head(5))

[data_loader] official train partition: (175341, 45)
[data_loader] official test  partition: (82332, 45)
[data_loader] combined corpus: (257673, 46)

Shape: (257673, 46)


Memory: 155.1 MB


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,sloss,dloss,sinpkt,dinpkt,sjit,djit,swin,stcpb,dtcpb,dwin,tcprtt,synack,ackdat,smean,dmean,trans_depth,response_body_len,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label,partition
0,1,0.121478,tcp,-,fin,6,4,258,172,74.087490,252,254,14158.942380,8495.365234,0,0,24.295600,8.375000,30.177547,11.830604,255,621772692,2202533631,255,0.000000,0.000000,0.000000,43,43,0,0,1,0,1,1,1,1,0,0,0,1,1,0,Normal,0,official_train
1,2,0.649902,tcp,-,fin,14,38,734,42014,78.473372,62,252,8395.112305,503571.312500,2,17,49.915000,15.432865,61.426934,1387.778330,255,1417884146,3077387971,255,0.000000,0.000000,0.000000,52,1106,0,0,43,1,1,1,1,2,0,0,0,1,6,0,Normal,0,official_train
2,3,1.623129,tcp,-,fin,8,16,364,13186,14.170161,62,252,1572.271851,60929.230470,1,6,231.875571,102.737203,17179.586860,11420.926230,255,2116150707,2963114973,255,0.111897,0.061458,0.050439,46,824,0,0,7,1,2,1,1,3,0,0,0,2,6,0,Normal,0,official_train
3,4,1.681642,tcp,ftp,fin,12,12,628,770,13.677108,62,252,2740.178955,3358.622070,1,3,152.876547,90.235726,259.080172,4991.784669,255,1107119177,1047442890,255,0.000000,0.000000,0.000000,52,64,0,0,1,1,2,1,1,3,1,1,0,2,1,0,Normal,0,official_train
4,5,0.449454,tcp,-,fin,10,6,534,268,33.373826,254,252,8561.499023,3987.059814,2,1,47.750333,75.659602,2415.837634,115.807000,255,2436137549,1977154190,255,0.128381,0.071147,0.057234,53,45,0,0,43,1,2,2,1,40,0,0,0,2,39,0,Normal,0,official_train


In [5]:
summary = pd.DataFrame({
    "dtype": corpus.dtypes.astype(str),
    "n_unique": corpus.nunique(),
    "n_missing": corpus.isna().sum(),
    "example": corpus.iloc[0],
})
display(summary)

,dtype,n_unique,n_missing,example
id,int64,175341,0,1
dur,float64,109945,0,0.121478
proto,object,133,0,tcp
service,object,13,0,-
state,object,11,0,fin
spkts,int64,646,0,6
dpkts,int64,627,0,4
sbytes,int64,9382,0,258
dbytes,int64,8653,0,172
rate,float64,115763,0,74.08749


## 3. Data quality

Three questions, in order of how much they affect the result: is anything
missing, is anything duplicated, and does anything leak the target?

In [6]:
print(f"Total missing values across the entire corpus: {corpus.isna().sum().sum()}")

# "No missing values" is not the same as "nothing is absent".
sentinel = (corpus["service"] == "-").sum()
silent = (corpus["dbytes"] == 0).sum()
print(f"service == '-'  (no L7 service identified) : {sentinel:,} ({sentinel/len(corpus):.1%})")
print(f"dbytes == 0     (destination never replied) : {silent:,} ({silent/len(corpus):.1%})")
print("\nBoth are REAL values, not absences. Neither is imputed anywhere in this project.")

Total missing values across the entire corpus: 0
service == '-'  (no L7 service identified) : 141,321 (54.8%)
dbytes == 0     (destination never replied) : 120,288 (46.7%)

Both are REAL values, not absences. Neither is imputed anywhere in this project.


In [7]:
predictors = [c for c in corpus.columns if c not in (*config.LEAKAGE_COLUMNS, "partition")]

exact = corpus.duplicated().sum()
on_predictors = corpus.duplicated(subset=predictors).sum()
conflicting = (corpus.groupby(predictors, dropna=False, observed=True)["label"]
               .nunique() > 1).sum()

print(f"Exact duplicate rows (including id) : {exact:,}")
print(f"Duplicate on the {len(predictors)} predictors  : {on_predictors:,} "
      f"({on_predictors/len(corpus):.1%})")
print(f"Vectors with CONTRADICTORY labels   : {conflicting:,}")

Exact duplicate rows (including id) : 0
Duplicate on the 42 predictors  : 103,989 (40.4%)
Vectors with CONTRADICTORY labels   : 414


### This is the most consequential finding in the dataset

**40.4% of the corpus repeats an earlier feature vector.** A random train/test
split therefore places byte-identical records on both sides, and a model that
memorises them is rewarded for it. Published UNSW-NB15 benchmarks that do not
deduplicate are reporting partly-memorised performance.

This project deduplicates **before** splitting. Notebook 03 shows the effect,
and the `keep_duplicates` ablation measures what it is worth.

The contradictory vectors are an irreducible noise floor — identical
measurements labelled both ways. No classifier can be right about both copies.

In [8]:
duplicate_mask = corpus.duplicated(subset=predictors, keep="first")
by_family = (corpus.assign(dup=duplicate_mask)
             .groupby("attack_cat", observed=True)["dup"]
             .agg(records="size", duplicated="sum", rate="mean")
             .sort_values("rate", ascending=False))
display(by_family.style.format({"records": "{:,}", "duplicated": "{:,}", "rate": "{:.1%}"}))

,records,duplicated,rate
attack_cat,,,
Generic,"58,871","51,566",87.6%
Backdoor,"2,329","1,746",75.0%
DoS,"16,353","12,163",74.4%
Analysis,"2,677","1,841",68.8%
Exploits,"44,525","18,719",42.0%
Reconnaissance,"13,987","5,341",38.2%
Fuzzers,"24,246","5,022",20.7%
Normal,"93,000","7,523",8.1%
Worms,174,10,5.7%


Duplication is **strongly class-dependent** — Generic is 87.6% duplicated,
Normal only 8.1%. That asymmetry is why deduplication changes the class balance
so much, and why leaving duplicates in place biases the result rather than
merely inflating the sample size.

## 4. Target and class distribution

In [9]:
counts = corpus["label"].value_counts().sort_index()
print("Binary target:")
for value, count in counts.items():
    name = "benign" if value == 0 else "attack"
    print(f"  {value} ({name:<6}) {count:>8,}  {count/len(corpus):.2%}")

print("\nAttack families:")
for name, count in corpus["attack_cat"].value_counts().items():
    print(f"  {name:<18} {count:>8,}  {count/len(corpus):.2%}")

Binary target:
  0 (benign)   93,000  36.09%
  1 (attack)  164,673  63.91%

Attack families:
  Normal               93,000  36.09%
  Generic              58,871  22.85%
  Exploits             44,525  17.28%
  Fuzzers              24,246  9.41%
  DoS                  16,353  6.35%
  Reconnaissance       13,987  5.43%
  Analysis              2,677  1.04%
  Backdoor              2,329  0.90%
  Shellcode             1,511  0.59%
  Worms                   174  0.07%


**Attacks are the MAJORITY class (63.9%).** This is the inverse of any real
network, where malicious flows are a fraction of a percent.

The consequence carries through every later result: **precision depends on the
base rate; recall and false-positive rate do not.** Any precision figure measured
on this corpus is optimistic for deployment.

## 5. Leakage audit

In [10]:
cross = pd.crosstab(corpus["attack_cat"], corpus["label"])
display(cross)

is_normal = corpus["attack_cat"].str.lower().eq("normal")
agreement = (is_normal == corpus["label"].eq(0)).mean()
print(f"\nattack_cat == 'Normal'  <=>  label == 0 : agreement = {agreement:.6f}")
print("\nPerfect agreement => attack_cat IS the target. Excluded as a predictor.")

label,0,1
attack_cat,,
Analysis,0,2677
Backdoor,0,2329
DoS,0,16353
Exploits,0,44525
Fuzzers,0,24246
Generic,0,58871
Normal,93000,0
Reconnaissance,0,13987
Shellcode,0,1511



attack_cat == 'Normal'  <=>  label == 0 : agreement = 1.000000

Perfect agreement => attack_cat IS the target. Excluded as a predictor.


In [11]:
print("Columns excluded from the feature matrix, and why:\n")
reasons = {
    "id": "synthetic row index; correlates with position in the capture, not with traffic",
    "attack_cat": "deterministic function of the target -> textbook target leakage",
    "label": "the target itself",
    "partition": "bookkeeping added by this project",
}
for column, reason in reasons.items():
    print(f"  {column:<12} {reason}")

from src import preprocessing
X, y = preprocessing.split_xy(corpus.head(100))
print(f"\nEnforced in one place (src.preprocessing.split_xy): "
      f"{X.shape[1]} predictors survive, target '{y.name}' separated.")

Columns excluded from the feature matrix, and why:

  id           synthetic row index; correlates with position in the capture, not with traffic
  attack_cat   deterministic function of the target -> textbook target leakage
  label        the target itself
  partition    bookkeeping added by this project



Enforced in one place (src.preprocessing.split_xy): 42 predictors survive, target 'label' separated.


### A caveat that is *not* conventional leakage

`sttl` and `dttl` are legitimate fields a real sensor observes — but in this
dataset the benign and attack generators ran on hosts with different initial TTL
values, making `sttl` close to a label proxy. That is **capture-artefact
leakage**, and it cannot be fixed by dropping a column without also discarding a
field a real sensor genuinely sees.

Notebook 02 quantifies it; notebook 06 measures its influence on the fitted
model with SHAP; and the `no_ttl` ablation retrains without it.

## 6. Data dictionary

In [12]:
dictionary_path = config.REPORTS_DIR / "data_dictionary.csv"
if not dictionary_path.exists():
    from src import document
    document.generate_dataset_documentation()

dictionary = pd.read_csv(dictionary_path)
print(f"{len(dictionary)} documented columns "
      f"({(dictionary['origin'] != 'UNSW-NB15 (published)').sum()} engineered by this project)\n")
display(dictionary[["feature_name", "category", "role", "possible_security_meaning"]].head(15))

61 documented columns (15 engineered by this project)



,feature_name,category,role,possible_security_meaning
0,id,Identifier,Excluded (row identifier),Row index only. Carries no network semantics a...
1,dur,Basic flow,Predictor (numeric),Session length. Scans and floods are sub-secon...
2,proto,Basic flow,Predictor (categorical),IP/transport protocol. Traffic on unusual prot...
3,service,Basic flow,Predictor (categorical),Application protocol identified by the analyse...
4,state,Basic flow,Predictor (categorical),Connection outcome. INT (no reply) dominates s...
5,spkts,Basic flow,Predictor (numeric),Outbound packet count. High counts with tiny p...
6,dpkts,Basic flow,Predictor (numeric),Inbound packet count. Zero means the target ne...
7,sbytes,Basic flow,Predictor (numeric),Outbound volume. The primary signal for data e...
8,dbytes,Basic flow,Predictor (numeric),Inbound volume. Large values with small reques...
9,rate,Basic flow,Predictor (numeric),Packets per second. Machine-generated attack t...


---

## Conclusions

| Finding | Consequence for the project |
|---|---|
| Both partitions verified against published row counts and SHA-256 | Dataset identity is confirmed, not assumed |
| Zero missing values, but `'-'` and `0` are meaningful | Nothing is imputed anywhere |
| 40.4% duplicate feature vectors, class-correlated | **Deduplicate before splitting** (notebook 03) |
| 414 contradictory vectors | Irreducible noise floor on achievable accuracy |
| Attacks are the majority class | Precision will not transfer to deployment |
| `attack_cat` determines `label` exactly | Excluded as a predictor at a single enforcement point |
| TTL fields look like label proxies | Quantified in notebook 02, measured in notebook 06, ablated in `src/train.py` |
| No timestamps, IPs or ports in the partitioned files | **No temporal holdout is possible** — drift cannot be measured |

**Next:** [`02_eda.ipynb`](02_eda.ipynb) — exploratory analysis on the training
split only.